In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
import umap
import hdbscan
import matplotlib.pyplot as plt
import pickle
from tqdm.notebook import tqdm

In [ ]:
cur_dir = os.getcwd().split('\\')

if cur_dir[-1] == 'notebooks':
    os.chdir("..")

from utils.data_load_utilities.data_loader import load_model_results
from utils.get_global_const import get_global_const
from utils.get_metrics import get_metrics
from utils.get_ranks import get_ranks_s, get_ranks
from methods.ADoE_method import *
from methods.k_nearest_methods import *
from methods.kmeans_methods import *
from methods.opt_methods import *
from methods.sparce_methods import *
from methods.entrophy_metods import *

from testing_pipeline.testing_pipeline_stats import *

from sklearn.linear_model import LassoCV
from sklearn.feature_selection import mutual_info_regression

from datetime import datetime
import re

import warnings
warnings.simplefilter('ignore')

In [ ]:
scores, datasets, models = get_global_const()
scores

In [ ]:
tsfresh_features = pd.read_csv(Path('data/datasets_features/tsfresh_important_features.csv'), index_col=0)
# tsfresh_features

In [ ]:
# chosen_datasets = tsfresh_features.Name.values
chosen_datasets = datasets
chosen_datasets[:5]

In [ ]:
features = pd.read_csv(Path('data/datasets_features/features.csv'), index_col=0)
features = features.set_index('Name')
features = features.loc[chosen_datasets, :]
features = features.reset_index()
features

In [ ]:
scores_aggr = {
    model: model_score[model_score["folds:"].isin(chosen_datasets)].reset_index(drop=True) for model, model_score in scores.items()
}

scores_aggr = {
    model: model_score.set_index("folds:").loc[chosen_datasets, :].reset_index() for model, model_score in scores_aggr.items()
}

scores_aggr = {
    model: model_score[model_score.columns[1:]].mean(axis=1) for model, model_score in scores_aggr.items()
}

scores_aggr = pd.DataFrame(scores_aggr)

scores_aggr

In [ ]:
prep_features = features.copy()
prep_features = prep_features.drop(columns=['Name'])
prep_features['size'] = np.log(prep_features['size'])
prep_features

In [ ]:
prep_tsfresh_features = tsfresh_features.copy()
prep_tsfresh_features = prep_tsfresh_features.drop(columns=['Name'])
prep_tsfresh_features

In [ ]:
nn_clusters = [2, 3, 4, 5, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 109]

In [ ]:
chosen_datasets = np.array(chosen_datasets)

In [ ]:
ranks = get_ranks_s(chosen_datasets, scores, datasets, return_ranks=True)
ranks_all = get_ranks_s(chosen_datasets, scores, datasets, return_ranks=False)

In [ ]:
# 1/0

In [ ]:
# ranks

In [ ]:
metrics = ['MAE', 'MSE', 'Kendall', 'Spearman', 'Pearson', 'Mutual information', 'Adjusted mutual information', 'Normalized mutual information', 'Cramers V', 'Point-Biserial Correlation', 'Phi Coefficient', 'Bloomquist Beta', 'Rank Correlation Ratio', 'Xi Correlation', 'Distance Correlation', 'Me', 'repr']

In [ ]:
def random_ind_metr(data: pd.DataFrame, indexes_list, sample_size: int, test_ind, datasets, iter=100, inner_iter=10):
    
    sum_metrics = None
    
    for i in range(iter):
        
        for _ in range(inner_iter):
        
            data_train = data.loc[indexes_list[i]]
            
            indxs = np.random.choice(data_train.index, sample_size, replace=False)
            
            repr_simple = chosen_datasets[[indxs]].squeeze()
        
            ranks_simple = get_ranks(repr_simple, ranks)
            metr = get_metrics(get_ranks(datasets[test_ind].squeeze(), ranks), ranks_simple)
            
            if sum_metrics is None:
                sum_metrics = metr.copy()
            else:
                for key in sum_metrics:
                    sum_metrics[key] += metr[key]

    averaged_metrics = {key: value / (iter * inner_iter) for key, value in sum_metrics.items()}

    return averaged_metrics
    

In [ ]:
targets = np.array(scores_aggr.mean(axis=1))

In [ ]:
tsfresh_features_full = pd.read_csv(Path('data/datasets_features/tsfresh_full_features.csv'), index_col=0)
prep_tsfresh_features_full = tsfresh_features_full.copy().set_index("Name").loc[chosen_datasets, :]
prep_tsfresh_features_full = prep_tsfresh_features_full.reset_index().drop(columns=['Name'])
prep_tsfresh_features_full = prep_tsfresh_features_full.dropna(axis=1)
prep_tsfresh_features_full = prep_tsfresh_features_full.drop(columns=prep_tsfresh_features_full.columns[(prep_tsfresh_features_full.abs() > 1e9).any()])
constant_features = [col for col in prep_tsfresh_features_full.columns if prep_tsfresh_features_full[col].nunique() <= 1]
prep_tsfresh_features_full = prep_tsfresh_features_full.drop(columns=constant_features)
prep_tsfresh_features_full

In [ ]:
meta_tsfresh = pd.read_csv(Path('data/datasets_features/apriori_meta_tsfresh_probe.csv'), index_col=0).reset_index(drop=True)
meta_tsfresh.head()

In [ ]:
meta_catch22 = pd.read_csv(Path('data/datasets_features/apriori_meta_catch22_probe.csv'), index_col=0).reset_index(drop=True)
meta_catch22.head()

In [ ]:
meta_summary = pd.read_csv(Path('data/datasets_features/apriori_meta_summary_probe.csv'), index_col=0).reset_index(drop=True)
meta_summary.head()

In [ ]:
ranks_aggr = pd.DataFrame(columns=chosen_datasets)

for d in chosen_datasets:
    ranks_aggr[d] = ranks[d].drop(columns=['model']).mean(axis=1)

ranks_aggr.transpose()

In [ ]:
ranks_aggr_rounded = ranks_aggr.rank(axis=0)
ranks_aggr_rounded[:5]

In [ ]:
tsf = pd.read_csv("data/datasets_features/dataset_level/tsfresh_dataset_mean.csv").sort_values("dataset").set_index("dataset").loc[chosen_datasets, :].reset_index().drop(columns=['dataset'])
c22 = pd.read_csv("data/datasets_features/dataset_level/catch22_dataset_mean.csv").sort_values("dataset").set_index("dataset").loc[chosen_datasets, :].reset_index().drop(columns=['dataset'])
mr  = pd.read_csv("data/datasets_features/dataset_level/minirocket_dataset_mean.csv").sort_values("dataset").set_index("dataset").loc[chosen_datasets, :].reset_index().drop(columns=['dataset'])
sm  = pd.read_csv("data/datasets_features/dataset_level/summary_dataset_mean.csv").sort_values("dataset").set_index("dataset").loc[chosen_datasets, :].reset_index().drop(columns=['dataset'])

In [ ]:
lm = pd.read_csv("data/datasets_features/landmarking_raw.csv").sort_values("dataset").set_index("dataset").loc[chosen_datasets, :].reset_index().drop(columns=['dataset'])

In [ ]:
tsf = tsf.dropna(axis=1)
tsf = tsf.drop(columns=tsf.columns[(tsf.abs() > 1e9).any()])
constant_features = [col for col in tsf.columns if tsf[col].nunique() <= 1]
tsf = tsf.drop(columns=constant_features)
tsf

In [ ]:
tsf.isna().values.any()

In [ ]:
data_rec = pd.read_csv(Path('data/rec_sys_datasets/Datafeatures_and_metrics.csv')).sort_values(by='Dataset')

In [ ]:
datasets_recsys = data_rec['Dataset'].unique()
datasets_recsys, datasets_recsys.shape

In [ ]:
data_rec.head()

In [ ]:
metrics_data = pd.read_csv(Path('data/rec_sys_datasets/add_data/metrics_ndcg_10.csv'))
metrics_data.head()

In [ ]:
temp = data_rec.groupby(["Method", "Dataset"])["Value"].mean().unstack("Dataset")
# temp = metrics_data.groupby(["Method", "Dataset"])["Value"].mean().unstack("Dataset")
ranks_aggr_recsys = temp.rank(method="min", ascending=False)
ranks_aggr_recsys.index.name = None
ranks_aggr_recsys.columns.name = None
# ranks_aggr_recsys = ranks_aggr_recsys.reset_index(drop=True)
ranks_aggr_recsys = ranks_aggr_recsys
ranks_aggr_recsys

In [ ]:
ranks_recsys = {}
for ds in datasets_recsys:
    df_ds = pd.DataFrame({
        "model": ranks_aggr_recsys.index,
    })
    for i in range(30):
        df_ds[str(i)] = ranks_aggr_recsys[ds].values
    ranks_recsys[ds] = df_ds
ranks_recsys

In [ ]:
ranks

In [ ]:
recsys_features = (data_rec.drop(columns=['Method', 'Value']).drop_duplicates(subset=['Dataset'])
                   .sort_values(by='Dataset').reset_index(drop=True))
recsys_features.head()

In [ ]:
scores_aggr

In [ ]:
ranks_aggr_recsys

In [ ]:
scores_aggr_recsys = data_rec.sort_values(by='Dataset').pivot(index='Dataset', columns='Method', values='Value').reset_index()
scores_aggr_recsys = scores_aggr_recsys.set_index('Dataset').reindex(recsys_features['Dataset']).reset_index()
scores_aggr_recsys = scores_aggr_recsys.drop(columns=['Dataset'])
recsys_features = recsys_features.drop(columns=['Dataset'])
scores_aggr_recsys.columns.name = None
scores_aggr_recsys

In [ ]:
models_recsys = scores_aggr_recsys.keys()
models_recsys

In [ ]:
scores_recsys = {}
for model in models_recsys:
    df_ds = pd.DataFrame({
        "folds:": ranks_aggr_recsys.columns,
    })
    for i in range(30):
        df_ds[str(i)] = scores_aggr_recsys[model].values
    scores_recsys[model] = df_ds
scores_recsys

In [ ]:
scores

In [ ]:
def a_optimality_ind(data,
                     sample_size: int,
                     model_list=None,
                     scale_data: bool = True,
                     iter: int = 100,
                     alpha: float = 1e-4,
                     random_state: int = 42,
                     **kwargs):

    X = np.asarray(data)

    if scale_data:
        X = StandardScaler().fit_transform(X)

    df = pd.DataFrame(X)
    
    return a_d_optimality_ind(df,
                              sample_size=sample_size,
                              optimality='a',
                              iter=iter,
                              alpha=alpha,
                              random_state=random_state,
                              return_ind=True)


def d_optimality_ind(data,
                     sample_size: int,
                     model_list=None,
                     scale_data: bool = True,
                     iter: int = 100,
                     alpha: float = 1e-4,
                     random_state: int = 42,
                     **kwargs):

    X = np.asarray(data)

    if scale_data:
        X = StandardScaler().fit_transform(X)

    df = pd.DataFrame(X)
    return a_d_optimality_ind(df,
                              sample_size=sample_size,
                              optimality='d',
                              iter=iter,
                              alpha=alpha,
                              random_state=random_state,
                              return_ind=True)

In [ ]:
metods_data_list = [
                    [rand_ind_method, recsys_features.values,
                     range(2, 21),
                     False,
                     False,
                     {}, 
                     False,
                     False,
                     'Random_RecSys_200_F'],
                    # [get_more_different_datasets,
                    #  meta_tsfresh.values,
                    #  range(2, 21),
                    #  False,
                    #  False,
                    #  {'scale_data': True},
                    #  False,
                    #  False,
                    #  'Cosine_Method_SF_TEST_D4'],
                    [get_more_different_datasets,
                     recsys_features.values,
                     range(2, 21),
                     False,
                     False,
                     {'scale_data': True},
                     False,
                     False, 
                     'Cosine_Method_RecSys_200_F'],
                    [k_means_ind,
                     recsys_features.values,
                     range(2, 21),
                     False,
                     False,
                     {'scale_data': True},
                     False,
                     False, 
                     'K-Means_RecSys_200_F'],
                    [get_more_different_datasets_euclid,
                     recsys_features.values,
                     range(2, 21),
                     False,
                     False,
                     {'scale_data': True},
                     False,
                     False, 
                     'EuclidMethod_RecSys_200_F'],
                    [a_optimality_ind,
                     recsys_features.values,
                     range(2, 21),
                     False,
                     False,
                     {'scale_data': True, 'iter': 5, 'alpha': 1e-4, 'random_state': 42},
                     False,
                     False, 
                     'A-opt_RecSys_200_F'],
                    [d_optimality_ind,
                     recsys_features.values,
                     range(2, 21),
                     False,
                     False,
                     {'scale_data': True, 'iter': 5, 'alpha': 1e-4, 'random_state': 42},
                     False,
                     False, 
                     'D-opt_RecSys_200_F'],
                    ]


In [ ]:
testing_pipeline(datasets_recsys, metods_data_list, ranks_recsys, scores_recsys, datasets_recsys, ratio=0.8, models_prefix='',
                 load_res=True, save_results=True, save_checpoints=True, num_models=9, test_iter=200)

In [ ]:
ranks_aggr_r = pd.DataFrame(columns=datasets_recsys)

for d in datasets_recsys:
    ranks_aggr_r[d] = ranks_recsys[d].drop(columns=['model']).mean(axis=1)


ranks_aggr_r = ranks_aggr_r.transpose()
ranks_aggr_r = ranks_aggr_r.reset_index(names='dataset').drop(columns=["dataset"])
ranks_aggr_r

In [ ]:
ranks_aggr_t = ranks_aggr.transpose().reset_index(drop=True)
ranks_aggr_t

In [ ]:
corr_matrix = np.corrcoef(ranks_aggr_r.values)
upper_triangle = corr_matrix[np.triu_indices_from(corr_matrix, k=1)]
mean_pairwise_corr = upper_triangle.mean()
mean_pairwise_corr

In [ ]:
corr_matrix = np.corrcoef(ranks_aggr_t.values)
upper_triangle = corr_matrix[np.triu_indices_from(corr_matrix, k=1)]
mean_pairwise_corr = upper_triangle.mean()
mean_pairwise_corr

In [ ]:
from scipy.stats import pearsonr, spearmanr

In [ ]:
def feature_dependence_analysis(
    df,
    target_col,
    random_state=42,
):

    feature_cols = [c for c in df.columns if c != target_col]

    X = df[feature_cols].copy()
    y = df[target_col].values

    results = []

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    mi = mutual_info_regression(
        X_scaled,
        y,
        random_state=random_state,
    )

    for i, feature in enumerate(feature_cols):
        x = df[feature].values

        pearson_corr, pearson_p = pearsonr(x, y)
        spearman_corr, spearman_p = spearmanr(x, y)

        results.append({
            "feature": feature,
            "pearson_corr": pearson_corr,
            "pearson_pvalue": pearson_p,
            "spearman_corr": spearman_corr,
            "spearman_pvalue": spearman_p,
            "mutual_information": mi[i],
        })

    res_df = pd.DataFrame(results)

    res_df = res_df.sort_values(by="mutual_information", ascending=False).reset_index(drop=True)

    return res_df

In [ ]:
metrics_recsys = scores_aggr_recsys.mean(axis=1)
metrics_recsys.shape

In [ ]:
results_recsys = feature_dependence_analysis(
    recsys_features.join(metrics_recsys.rename('metric')),
    target_col="metric"
)

In [ ]:
results_recsys

In [ ]:
selected_features_recsys = results_recsys.sort_values('mutual_information', ascending=False).head(10)
selected_features_recsys.head()

In [ ]:
metods_data_list = [
                    [rand_ind_method, recsys_features[selected_features_recsys['feature'].to_list()].values,
                     range(2, 21),
                     False,
                     False,
                     {}, 
                     False,
                     False,
                     'Random_RecSys_200_FeatSelect'],
                    # [get_more_different_datasets,
                    #  meta_tsfresh.values,
                    #  range(2, 21),
                    #  False,
                    #  False,
                    #  {'scale_data': True},
                    #  False,
                    #  False,
                    #  'Cosine_Method_SF_TEST_D4'],
                    [get_more_different_datasets,
                     recsys_features[selected_features_recsys['feature'].to_list()].values,
                     range(2, 21),
                     False,
                     False,
                     {'scale_data': True},
                     False,
                     False, 
                     'Cosine_Method_RecSys_200_FeatSelect'],
                    [k_means_ind,
                     recsys_features[selected_features_recsys['feature'].to_list()].values,
                     range(2, 21),
                     False,
                     False,
                     {'scale_data': True},
                     False,
                     False, 
                     'K-Means_RecSys_200_FeatSelect'],
                    [get_more_different_datasets_euclid,
                     recsys_features[selected_features_recsys['feature'].to_list()].values,
                     range(2, 21),
                     False,
                     False,
                     {'scale_data': True},
                     False,
                     False, 
                     'EuclidMethod_RecSys_200_FeatSelect'],
                    [a_optimality_ind,
                     recsys_features[selected_features_recsys['feature'].to_list()].values,
                     range(2, 21),
                     False,
                     False,
                     {'scale_data': True, 'iter': 5, 'alpha': 1e-4, 'random_state': 42},
                     False,
                     False, 
                     'A-opt_RecSys_200_FeatSelect'],
                    [d_optimality_ind,
                     recsys_features[selected_features_recsys['feature'].to_list()].values,
                     range(2, 21),
                     False,
                     False,
                     {'scale_data': True, 'iter': 5, 'alpha': 1e-4, 'random_state': 42},
                     False,
                     False, 
                     'D-opt_RecSys_200_FeatSelect'],
                    ]


In [ ]:
testing_pipeline(datasets_recsys, metods_data_list, ranks_recsys, scores_recsys, datasets_recsys, ratio=0.8, models_prefix='',
                 load_res=True, save_results=True, save_checpoints=True, num_models=9, test_iter=200)